#### Load data

In [92]:
import pandas as pd
import numpy as np

date_str_1 = "2025-11-01"
date_str_2 = "2025-11-15"
date_str_3 = "2025-11-29"

date_str_active = date_str_3

current_snapshot_date = pd.Timestamp(date_str_active, tz="UTC")
current_snapshot_date_str = date_str_active

In [93]:

# =========================
# 1. Load data
# =========================
content = pd.read_csv("Data/Content.csv")
solutions = pd.read_csv("Data/Solutions.csv")

# =========================
# 2. Parse dates
# =========================
content["CreatedDate"] = pd.to_datetime(content["CreatedDate"], utc=True, errors="coerce")
content["LastAllowedDate"] = pd.to_datetime(content["LastAllowedDate"], utc=True, errors="coerce")
solutions["CreatedDate"] = pd.to_datetime(solutions["CreatedDate"], utc=True, errors="coerce")

# =========================
# 3. Define simulation date
# =========================
snapshot_date = current_snapshot_date

# =========================
# 4. Filter content:
# only assignments (ContentType = 0)
# and only those available by snapshot date
# =========================
content_snapshot = content[
    (content["CreatedDate"] <= snapshot_date) &
    (content["ContentType"] == 0)
].copy()

# =========================
# 5. Filter solutions by date
# =========================
solutions_snapshot = solutions[
    solutions["CreatedDate"] <= snapshot_date
].copy()

# =========================
# 6. Keep only solutions for valid assignments
# =========================
content_cols = ["Id", "CourseId", "CreatedDate", "LastAllowedDate", "Name"]
content_small = content_snapshot[content_cols].copy()

merged = solutions_snapshot.merge(
    content_small,
    left_on="ContentId",
    right_on="Id",
    how="inner",
    suffixes=("_solution", "_content")
)

# =========================
# 7. Deduplicate:
# keep latest submission per student per assignment
# =========================
merged = merged.sort_values(["UserId", "ContentId", "CreatedDate_solution"])
merged_latest = merged.groupby(["UserId", "ContentId"], as_index=False).tail(1).copy()

# =========================
# 8. Convert grades
# =========================
merged_latest["Grade"] = pd.to_numeric(merged_latest["Grade"], errors="coerce")

# =========================
# 9. Build student list
# =========================
student_ids = sorted(merged_latest["UserId"].dropna().unique())
student_snapshot = pd.DataFrame({"student_id": student_ids})

# =========================
# 10. Total assignments
# =========================
total_assignments_available = content_small["Id"].nunique()
student_snapshot["total_assignments_available"] = total_assignments_available

# =========================
# 11. Submitted assignments
# =========================
submitted_counts = (
    merged_latest.groupby("UserId")["ContentId"]
    .nunique()
    .reset_index(name="submitted_assignments")
    .rename(columns={"UserId": "student_id"})
)

# =========================
# 12. Late submissions
# =========================
merged_latest["is_late"] = (
    merged_latest["LastAllowedDate"].notna() &
    (merged_latest["CreatedDate_solution"] > merged_latest["LastAllowedDate"])
)

late_counts = (
    merged_latest.groupby("UserId")["is_late"]
    .sum()
    .reset_index(name="late_submissions")
    .rename(columns={"UserId": "student_id"})
)

# =========================
# 13. Average grade
# =========================
avg_grade = (
    merged_latest.groupby("UserId")["Grade"]
    .mean()
    .reset_index(name="average_grade")
    .rename(columns={"UserId": "student_id"})
)

# =========================
# 14. Recent average grade (last 3)
# =========================
recent_grades = merged_latest.sort_values(
    ["UserId", "CreatedDate_solution"], ascending=[True, False]
).copy()

recent_grades["rank"] = recent_grades.groupby("UserId").cumcount() + 1
recent_top3 = recent_grades[recent_grades["rank"] <= 3]

recent_avg_grade = (
    recent_top3.groupby("UserId")["Grade"]
    .mean()
    .reset_index(name="recent_average_grade")
    .rename(columns={"UserId": "student_id"})
)

# =========================
# 15. Last submission + recency
# =========================
last_submission = (
    merged_latest.groupby("UserId")["CreatedDate_solution"]
    .max()
    .reset_index(name="last_submission_date")
    .rename(columns={"UserId": "student_id"})
)

last_submission["days_since_last_submission"] = (
    snapshot_date - last_submission["last_submission_date"]
).dt.days

# =========================
# 16. Assemble snapshot
# =========================
student_snapshot = student_snapshot.merge(submitted_counts, on="student_id", how="left")
student_snapshot = student_snapshot.merge(late_counts, on="student_id", how="left")
student_snapshot = student_snapshot.merge(avg_grade, on="student_id", how="left")
student_snapshot = student_snapshot.merge(recent_avg_grade, on="student_id", how="left")
student_snapshot = student_snapshot.merge(last_submission, on="student_id", how="left")

# =========================
# 17. Fill missing values
# =========================
student_snapshot["submitted_assignments"] = student_snapshot["submitted_assignments"].fillna(0).astype(int)
student_snapshot["late_submissions"] = student_snapshot["late_submissions"].fillna(0).astype(int)

student_snapshot["average_grade"] = student_snapshot["average_grade"].round(2)
student_snapshot["recent_average_grade"] = student_snapshot["recent_average_grade"].round(2)

# =========================
# 18. Submission rate
# =========================
student_snapshot["submission_rate"] = (
    student_snapshot["submitted_assignments"] / student_snapshot["total_assignments_available"]
).round(3)

# =========================
# 19. Final order
# =========================
student_snapshot = student_snapshot[
    [
        "student_id",
        "total_assignments_available",
        "submitted_assignments",
        "submission_rate",
        "late_submissions",
        "average_grade",
        "recent_average_grade",
        "last_submission_date",
        "days_since_last_submission",
    ]
]

student_snapshot = student_snapshot.sort_values("student_id").reset_index(drop=True)

# =========================
# 20. Preview
# =========================
print("Student snapshot rows:", len(student_snapshot))
print(student_snapshot.head(20))

Student snapshot rows: 30
    student_id  total_assignments_available  submitted_assignments  \
0            9                           26                     24   
1           10                           26                     23   
2           11                           26                     25   
3           12                           26                     24   
4           13                           26                     25   
5           14                           26                     24   
6           15                           26                     26   
7           16                           26                     26   
8           17                           26                     25   
9           18                           26                     26   
10          19                           26                     26   
11          20                           26                     25   
12          21                           26                     

In [94]:
# =========================
# Load students (only learners)
# =========================
users = pd.read_csv("Data/Users.csv")

# Keep only students (UserTypeId = 1)
students = users[users["UserTypeId"] == 1].copy()

# Select needed columns
students = students[
    ["Id", "Email", "FirstName", "LastName"]
].copy()

# Rename for consistency
students = students.rename(columns={"Id": "student_id"})

# Optional: sort
students = students.sort_values("student_id").reset_index(drop=True)

# Preview
print("Students:", len(students))
print(students.head(10))

Students: 31
   student_id                        Email FirstName LastName
0           7      john_doe@pseudomail.com      John      Doe
1           9     dan_cohen@pseudomail.com       Dan    Cohen
2          10      noa_levy@pseudomail.com       Noa     Levy
3          11  itay_mizrahi@pseudomail.com      Itay  Mizrahi
4          12    yael_biton@pseudomail.com      Yael    Biton
5          13   omer_peretz@pseudomail.com      Omer   Peretz
6          14    maya_david@pseudomail.com      Maya    David
7          15  lior_shapira@pseudomail.com      Lior  Shapira
8          16  tomer_azulay@pseudomail.com     Tomer   Azulay
9          17     shir_katz@pseudomail.com      Shir     Katz


In [95]:
# =========================
# Merge student info (names)
# =========================
student_snapshot = student_snapshot.merge(
    students,
    on="student_id",
    how="left"
)

#### Pre-LLM

Low performance

In [96]:
# =========================
# Low performance detector
# Rule: average_grade < 50
# =========================
low_performance_students = (
    student_snapshot[student_snapshot["average_grade"] < 50]
    .copy()
    .sort_values(["average_grade", "recent_average_grade"], ascending=[True, True])
    .reset_index(drop=True)
)

print("Low performance students:", len(low_performance_students))
print(
    low_performance_students[
        [
            "FirstName",
            "LastName",
            "average_grade",
            "recent_average_grade",
            "submission_rate",
            "late_submissions",
            "days_since_last_submission",
        ]
    ]
)

Low performance students: 3
  FirstName LastName  average_grade  recent_average_grade  submission_rate  \
0      Itay  Mizrahi           9.84                 14.33            0.962   
1       Dan    Cohen          12.75                 17.33            0.923   
2       Noa     Levy          13.04                 20.33            0.885   

   late_submissions  days_since_last_submission  
0                13                           2  
1                 8                           6  
2                 9                           6  


Declining performance

In [97]:
# =========================
# Declining performance detector
# Rule:
# recent_average_grade is at least 15 points lower than average_grade
# =========================
declining_performance_students = (
    student_snapshot[
        (student_snapshot["recent_average_grade"].notna()) &
        ((student_snapshot["average_grade"] - student_snapshot["recent_average_grade"]) >= 15)
    ]
    .copy()
)

declining_performance_students["grade_drop"] = (
    declining_performance_students["average_grade"] -
    declining_performance_students["recent_average_grade"]
).round(2)

declining_performance_students = (
    declining_performance_students
    .sort_values(["grade_drop", "recent_average_grade"], ascending=[False, True])
    .reset_index(drop=True)
)

print("Declining performance students:", len(declining_performance_students))
print(
    declining_performance_students[
        [
            "FirstName",
            "LastName",
            "average_grade",
            "recent_average_grade",
            "grade_drop",
            "submission_rate",
            "late_submissions",
            "days_since_last_submission",
        ]
    ]
)

Declining performance students: 1
  FirstName LastName  average_grade  recent_average_grade  grade_drop  \
0     Rotem    Golan          70.64                  54.0       16.64   

   submission_rate  late_submissions  days_since_last_submission  
0            0.962                 2                           2  


Low submission rate

In [98]:
# =========================
# Low engagement detector
# Rule: submission_rate < 0.90
# =========================
low_submission_students = (
    student_snapshot[student_snapshot["submission_rate"] < 0.8]
    .copy()
    .sort_values(
        ["submission_rate", "submitted_assignments", "late_submissions"],
        ascending=[True, True, False]
    )
    .reset_index(drop=True)
)

print("Low submission students:", len(low_submission_students))
print(
    low_submission_students[
        [
            "FirstName",
            "LastName",
            "submitted_assignments",
            "total_assignments_available",
            "submission_rate",
            "late_submissions",
            "average_grade",
            "recent_average_grade",
            "days_since_last_submission",
        ]
    ]
)

Low submission students: 0
Empty DataFrame
Columns: [FirstName, LastName, submitted_assignments, total_assignments_available, submission_rate, late_submissions, average_grade, recent_average_grade, days_since_last_submission]
Index: []


Top students

In [99]:
# =========================
# Top students detector
# Rule:
# average_grade >= 85
# =========================
top_students = (
    student_snapshot[student_snapshot["average_grade"] >= 85]
    .copy()
    .sort_values(
        ["average_grade", "recent_average_grade", "submission_rate"],
        ascending=[False, False, False]
    )
    .reset_index(drop=True)
)

print("Top students:", len(top_students))
print(
    top_students[
        [
            "FirstName",
            "LastName",
            "average_grade",
            "recent_average_grade",
            "submission_rate",
            "late_submissions",
            "days_since_last_submission",
        ]
    ]
)

Top students: 3
  FirstName  LastName  average_grade  recent_average_grade  submission_rate  \
0     Sivan   Elkayam          96.31                 96.67            1.000   
1        Or  Turgeman          92.56                 95.33            0.962   
2       Nir     Hazan          88.60                 97.00            0.962   

   late_submissions  days_since_last_submission  
0                 0                           1  
1                 1                           1  
2                 2                           1  


In [100]:
# =========================
# Save outputs for agent notebook
# =========================
student_snapshot.to_csv(f"ProcessedData/student_snapshot_{current_snapshot_date_str}.csv", index=False)

low_performance_students.to_csv(f"ProcessedData/low_performance_students_{current_snapshot_date_str}.csv", index=False)
declining_performance_students.to_csv(f"ProcessedData/declining_performance_students_{current_snapshot_date_str}.csv", index=False)
low_submission_students.to_csv(f"ProcessedData/low_submission_students_{current_snapshot_date_str}.csv", index=False)
top_students.to_csv(f"ProcessedData/top_students_{current_snapshot_date_str}.csv", index=False)

print("Saved all snapshot and detector outputs.")

Saved all snapshot and detector outputs.
